# Desarrollo de un Problema de Planificación

**Actividad — Individual**

**Maestría en Inteligencia Artificial**  
**Materia:** Razonamiento y Planificación Automática

**Profesora:** _Adriana Cervantes Castillo_

**Autor:** _Omar Joel Montemayor Charles_  
**Matrícula:** _5931346_  
**Institución:** _Universidad Internacional de La Rioja_  

**Fecha de entrega:** 24 de mayo de 2026


---

## Resumen

La actividad documenta la modelización en **PDDL** de un problema de transporte mineral
resuelto por un robot rover. Se formaliza un dominio reutilizable y tres instancias con
complejidad creciente: escenario original (dos minerales, cinco localidades), extensión con
tercer mineral y laboratorio secundario, y variante con dos rovers y zona bloqueada. Los
planes se obtienen con **Pyperplan** (Alkhazraji et al., 2020). Adicionalmente se documenta
el procedimiento para ejecutar **Scorpion** (Seipp et al., 2020), ganador del
*Optimal Track* de la IPC 2018, mediante Singularity/Apptainer; debido a la ausencia de
entorno GNU/Linux en el equipo utilizado (Windows sin WSL2), la ejecución contenedorizada
no se completa y se reporta evidencia del intento.

---



## 1. Contexto: IPC 2018 y planificador ganador

La *International Planning Competition 2018 – Classical Tracks* (IPC 2018) es organizada
por ICAPS para evaluar planificadores automáticos sobre dominios PDDL estandarizados
(ICAPS, s. f.). El primer lugar del *Optimal Track* fue **Scorpion**, variante de Fast
Downward (Helmert, 2006) que incorpora particiones de costos saturados y heurísticas
abstractas (Seipp et al., 2020). Su distribución oficial se realizó mediante imágenes
**Singularity** (Kurtzer et al., 2017), cuya continuación open-source es **Apptainer**
(Apptainer Project, 2024). La ejecución del contenedor requiere un sistema GNU/Linux
nativo o vía WSL2.



---

## 2. Descripción del problema

Un robot rover debe transportar dos minerales excavados al **laboratorio de análisis**
(Localidad 5). El rover parte de la **Localidad 3**, puede cargar un único mineral
simultáneamente y debe respetar las siguientes restricciones topológicas:

| Conexión | Tipo |
|---|---|
| Localidad 3 ↔ Localidad 1 | Bidireccional |
| Localidad 3 → Localidad 2 | Unidireccional |
| Localidad 2 → Localidad 4 | Unidireccional |
| Localidad 3 ↔ Localidad 4 | Bidireccional |
| Localidad 4 ↔ Localidad 5 | Bidireccional |

```
        [loc1]
         ↑↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5 = LAB]
   ↓              ↑
   └──────────────┘
```



---

## 3. Dominio PDDL — `domain.pddl`

El dominio define tres tipos (`rover`, `localidad`, `mineral`) y tres acciones —**mover**,
**recoger** y **entregar**— con sus precondiciones y efectos. La asimetría del terreno se
representa con el predicado unidireccional `(camino ?l1 ?l2)` (McDermott et al., 1998).


In [ ]:
(define (domain rover-minerales)
  (:requirements :typing :negative-preconditions)

  (:types
    localidad mineral rover - object
  )

  (:predicates
    (en-rover ?r - rover ?l - localidad)        ; el rover está en localidad l
    (en-mineral ?m - mineral ?l - localidad)    ; el mineral m está en localidad l
    (transportando ?r - rover ?m - mineral)     ; el rover lleva el mineral m
    (laboratorio ?l - localidad)                ; localidad l tiene laboratorio
    (analizado ?m - mineral)                    ; el mineral m ya fue analizado
    (camino ?l1 - localidad ?l2 - localidad)    ; existe camino de l1 a l2
    (manos-libres ?r - rover)                   ; el rover no carga nada
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Mover el rover de una localidad a otra
  ; ------------------------------------------------------------------
  (:action mover
    :parameters (?r - rover ?desde - localidad ?hacia - localidad)
    :precondition (and
      (en-rover ?r ?desde)
      (camino ?desde ?hacia)
    )
    :effect (and
      (not (en-rover ?r ?desde))
      (en-rover ?r ?hacia)
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Recoger un mineral en la localidad donde está el rover
  ; ------------------------------------------------------------------
  (:action recoger
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (en-mineral ?m ?l)
      (manos-libres ?r)
    )
    :effect (and
      (transportando ?r ?m)
      (not (en-mineral ?m ?l))
      (not (manos-libres ?r))
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Entregar mineral en el laboratorio
  ; ------------------------------------------------------------------
  (:action entregar
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (transportando ?r ?m)
      (laboratorio ?l)
    )
    :effect (and
      (not (transportando ?r ?m))
      (manos-libres ?r)
      (analizado ?m)
    )
  )
)


---

## 4. Problema 1: escenario original

Rover inicial en Localidad 3 con manos libres; `mineral_1` en Localidad 1; `mineral_2`
en Localidad 2; laboratorio en Localidad 5. La meta requiere que ambos minerales estén
en estado `analizado`.


In [ ]:
(define (problem rover-problema1)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 - localidad
    mineral1 mineral2 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    ;   loc3 <-> loc1  (bidireccional)
    (camino loc3 loc1)
    (camino loc1 loc3)
    ;   loc3 -> loc2   (una dirección)
    (camino loc3 loc2)
    ;   loc2 -> loc4   (una dirección)
    (camino loc2 loc4)
    ;   loc3 <-> loc4  (bidireccional)
    (camino loc3 loc4)
    (camino loc4 loc3)
    ;   loc4 <-> loc5  (bidireccional)
    (camino loc4 loc5)
    (camino loc5 loc4)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
    )
  )
)


### 4.1 Plan obtenido con Pyperplan

```bash
pyperplan -H hff -s astar domain.pddl problem1.pddl
```



**Análisis del plan.** El plan contiene 13 acciones: el rover viaja a `loc1`, recoge
`mineral_1`, retorna por `loc3` (no existe camino directo `loc1 → loc4`) y lo entrega en
`loc5`; recupera manos libres, regresa a `loc3`, avanza a `loc2` y recoge `mineral_2`, lo
traslada via `loc2 → loc4 → loc5` y lo entrega. La longitud óptima refleja tanto la
restricción de carga unitaria como la asimetría del grafo.



---

## 5. Problema 2: tercer mineral y laboratorio secundario

Se incorpora una **Localidad 6** (con `mineral_3`) y una **Localidad 7** (laboratorio
secundario), conectadas mediante `loc5 → loc6` y `loc6 ↔ loc7`. La meta exige los tres
minerales analizados; cualquiera de los dos laboratorios puede recibir entregas.

```
[loc1] ↔ [loc3] ↔ [loc4] ↔ [loc5=LAB1] → [loc6]
           ↓          ↑                      ↔
         [loc2] ───────┘                   [loc7=LAB2]
```


In [ ]:
(define (problem rover-problema2)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 loc6 loc7 - localidad
    mineral1 mineral2 mineral3 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc6)

    ; Laboratorios
    (laboratorio loc5)
    (laboratorio loc7)

    ; Red de caminos (original)
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevas conexiones
    (camino loc5 loc6)    ; una sola dirección
    (camino loc6 loc7)    ; bidireccional
    (camino loc7 loc6)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
    )
  )
)


### 5.1 Plan obtenido con Pyperplan

```bash
pyperplan -H hff -s astar domain.pddl problem2.pddl
```



**Análisis del plan.** Tras entregar `mineral_2` en `loc5`, el rover encadena
`loc5 → loc6 → loc7` para depositar `mineral_3` en el laboratorio secundario, evitando
un retorno innecesario. Esto confirma que la heurística `hff` explota el predicado
`(laboratorio ?l)` sobre múltiples nodos y que el dominio admite nuevas instancias sin
modificación (McDermott et al., 1998).



---

## 6. Problema 3: zona bloqueada y dos rovers

Se añaden: un **segundo rover** (`rover2`) en Localidad 4; una **Localidad 8** accesible
solo mediante `loc1 → loc8 → loc4` (ciclo dirigido); `mineral_3` en `loc1` y `mineral_4`
en `loc8`. La meta exige los cuatro minerales analizados.

```
        [loc1] → [loc8]
         ↑↓          ↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5=LAB]
   ↓              ↑
   └──────────────┘
```


In [ ]:
(define (problem rover-problema3)
  (:domain rover-minerales)

  (:objects
    rover1 rover2 - rover
    loc1 loc2 loc3 loc4 loc5 loc8 - localidad
    mineral1 mineral2 mineral3 mineral4 - mineral
  )

  (:init
    ; Posiciones iniciales
    (en-rover rover1 loc3)
    (manos-libres rover1)
    (en-rover rover2 loc4)
    (manos-libres rover2)

    ; Minerales
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc1)   ; segundo mineral en loc1
    (en-mineral mineral4 loc8)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevos caminos hacia loc8
    (camino loc1 loc8)     ; una sola dirección
    (camino loc8 loc4)     ; salida directa a loc4
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
      (analizado mineral4)
    )
  )
)


### 6.1 Plan obtenido con Pyperplan

```bash
pyperplan -H hff -s astar domain.pddl problem3.pddl
```



**Análisis del plan.** El ciclo `loc1 → loc8 → loc4` permite recolectar `mineral_4` en un
único pase. Los dos minerales en `loc1` obligan a dos operaciones de `recoger` separadas
por una entrega, dado el predicado `manos-libres`. El planificador distribuye el trabajo
entre los dos rovers, reduciendo la longitud del plan respecto a un agente único, sin que
el dominio incluya predicados explícitos de coordinación.



---

## 7. Ejecución del planificador Scorpion (IPC 2018 Optimal Track)

### 7.a Vía oficial: Singularity / Apptainer

La distribución oficial de **Scorpion** es una imagen Singularity publicada en el
repositorio de la IPC 2018 (IPC 2018, s. f.). El comando de ejecución es:

```bash
singularity run --bind $PWD:/ext scorpion.img \
    /ext/domain.pddl /ext/p01.pddl \
    --plan-file /ext/sas_plan
```

El binario equivalente con Apptainer es idéntico salvo el nombre (`apptainer run …`).
Ambos requieren núcleo Linux nativo o WSL2.

### 7.b Alternativa: compilación desde fuente

```bash
git clone https://github.com/jendrikseipp/scorpion.git
cd scorpion
./build.py
./fast-downward.py --alias seq-opt-scorpion ../domain.pddl ../p01.pddl
```

También requiere toolchain GNU/Linux (CMake ≥ 3.16, GCC ≥ 9, Python 3.7+).

### 7.c Evidencia del intento

El equipo empleado opera bajo **Windows 11** sin GNU/Linux (ni WSL2); por tanto, no es
posible completar la ejecución. La siguiente captura documenta el intento realizado:

![Intento de ejecución de Scorpion en Windows](capturas/scorpion_intento.png)



---

## 8. Tarea Snake (problema 1) de IPC 2018

El dominio **Snake** del *Classical Optimal Track* modela un agente segmentado sobre una
cuadrícula; al consumir comida el segmento crece. Usa tipos `location` y `direction`, con
predicados clave `(is-goal ?l)`, `(connected ?l1 ?l2 ?d)`, `(occupied ?l)`,
`(snake-head ?l)` y `(snake-tail ?l)`. El **problema 1** define una cuadrícula reducida
y sirve como punto de validación habitual para planificadores óptimos. La ejecución se
realizaría con el comando de la Sección 7.a.



---

## 9. Conclusiones

**PDDL** separa dominio e instancia, permitiendo reutilizar el mismo `domain.pddl` para
los tres problemas planteados. La asimetría del terreno se modela mediante predicados
unidireccionales `(camino ?l1 ?l2)`, y la cooperación multi-agente emerge de las variables
`?r - rover` sin predicados explícitos de coordinación.

Los planes obtenidos con **Pyperplan** confirman que el modelo es satisfacible y coherente
con la topología del problema. Aunque la ejecución de **Scorpion** no pudo completarse por
ausencia de entorno GNU/Linux, se documenta el procedimiento reproducible tanto vía
imagen Singularity/Apptainer como mediante compilación desde fuente.



---

## Referencias

Alkhazraji, Y., Frorath, M., Grützner, M., Helmert, M., Liebetraut, T.,
Mattmüller, R., Ortlieb, M., Seipp, J., Springenberg, T., Stahl, P., & Wülfing, J.
(2020). *Pyperplan* (Versión 2.1) [Software]. GitHub.
https://github.com/aibasel/pyperplan

Apptainer Project. (2024). *Apptainer user documentation*.
https://apptainer.org/docs/

Ghallab, M., Nau, D., & Traverso, P. (2004). *Automated planning: Theory and
practice*. Morgan Kaufmann.

Helmert, M. (2006). The Fast Downward planning system. *Journal of Artificial
Intelligence Research*, *26*, 191–246. https://doi.org/10.1613/jair.1705

International Conference on Automated Planning and Scheduling. (s. f.).
*Competitions*. Recuperado el 14 de mayo de 2026, de
https://www.icaps-conference.org/competitions/

International Planning Competition. (s. f.). *IPC 2018 — Classical Tracks*.
Recuperado el 14 de mayo de 2026, de https://ipc2018-classical.bitbucket.io/

Kurtzer, G. M., Sochat, V., & Bauer, M. W. (2017). Singularity: Scientific
containers for mobility of compute. *PLOS ONE*, *12*(5), e0177459.
https://doi.org/10.1371/journal.pone.0177459

McDermott, D., Ghallab, M., Howe, A., Knoblock, C., Ram, A., Veloso, M., Weld, D.,
& Wilkins, D. (1998). *PDDL — The Planning Domain Definition Language* (Tech.
Rep. CVC TR-98-003/DCS TR-1165). Yale Center for Computational Vision and
Control.

Seipp, J. (s. f.). *Scorpion planner* [Repositorio de código]. GitHub. Recuperado
el 14 de mayo de 2026, de https://github.com/jendrikseipp/scorpion

Seipp, J., Keller, T., & Helmert, M. (2020). Saturated cost partitioning for
optimal classical planning. *Journal of Artificial Intelligence Research*, *67*,
129–167. https://doi.org/10.1613/jair.1.11673

Sylabs. (s. f.). *Singularity admin guide — Installing Singularity*. Recuperado
el 14 de mayo de 2026, de
https://docs.sylabs.io/guides/3.5/admin-guide/installation.html
